# Practical 1 — Uber Fare Prediction 🚕
### Machine Learning | Beginner-friendly practical notebook

**Aim:** Predict an Uber ride fare using trip distance, passenger count, pickup hour, and year. Compare **Linear Regression** and **Random Forest Regression**.

**How to use:** Keep `uber.csv` in the same folder as this notebook. Run each code cell from top to bottom using **Shift + Enter**. Read the explanation above each cell before running it.

**What you will learn:** Load → Clean → Distance → Outliers → Correlation → Split → Train → Evaluate.

> **Important:** These fares are in the dataset's currency (USD). The model is for learning; real-world Uber pricing depends on more than these features.

## 0. Install libraries (only if needed)

**Library** = ready-made Python code that helps us do a task.

- `pandas`: works with tables like Excel.
- `numpy`: mathematical calculations.
- `matplotlib` and `seaborn`: graphs.
- `scikit-learn`: machine learning.

If you get `ModuleNotFoundError`, uncomment and run the next cell once. Then restart the kernel if VS Code asks.

In [ ]:
# Remove the # from the next line ONLY if libraries are missing.
# %pip install numpy pandas matplotlib seaborn scikit-learn

## 1. Import libraries

`import` means **bring a toolbox into our program**. `as np` or `as pd` gives a short nickname.

**Viva:** Why Scikit-learn? → It provides ready-made ML algorithms and evaluation tools.

In [ ]:
# Libraries for calculations, data and graphs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tools for splitting data, training models and checking results
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

## 2. Load the dataset

- `read_csv()` opens a CSV file.
- `df` means **DataFrame**: a table of rows and columns.
- `head()` displays the first five rows.
- `shape` shows **(number of rows, number of columns)**.

**Remember:** CSV → DataFrame → Preview.

In [ ]:
# Read the Uber trip dataset (keep uber.csv beside this notebook)
df = pd.read_csv("uber.csv")
print("Dataset size:", df.shape)
print(df.head())

## 3. Clean missing data and unnecessary columns

- `key` is a trip identifier; it is not useful for our simple model.
- `Unnamed: 0` is an extra index column.
- `errors="ignore"` means don't crash if one of those columns is missing.
- `dropna()` removes rows that have missing values.

**Viva:** Why clean data? → Missing or incorrect data can cause training errors and poor predictions.

In [ ]:
# Remove columns we do not need
df = df.drop(columns=["Unnamed: 0", "key"], errors="ignore")

# Remove rows with missing values
df = df.dropna()
print("Rows after removing missing data:", len(df))

## 4. Check that coordinates are valid

**Latitude** tells us north/south position; valid range is **−90 to +90** degrees.
**Longitude** tells us east/west position; valid range is **−180 to +180** degrees.

Some dataset entries have impossible coordinates. We remove them before calculating distance.

`&` means **AND**: all conditions must be true. `between(a, b)` checks whether values are within the range.

In [ ]:
# Keep rows with valid pickup and drop-off coordinates
df = df[
    df["pickup_latitude"].between(-90, 90) &
    df["dropoff_latitude"].between(-90, 90) &
    df["pickup_longitude"].between(-180, 180) &
    df["dropoff_longitude"].between(-180, 180)
].copy()
print("Rows with valid coordinates:", len(df))

## 5. Haversine formula — calculate trip distance 🌍

### Why do we need it?

The dataset has **pickup and drop-off coordinates**, but no distance column. Haversine estimates the **shortest distance along Earth's curved surface** between those points (not road distance).

### Understand every word

- `lat1` = pickup latitude; `lat2` = drop-off latitude.
- `dlat` = difference in latitude; `dlon` = difference in longitude.
- **Degrees → radians:** `np.radians()` converts angles into the unit used by `sin()` and `cos()`.
- `sin` and `cos` are mathematical functions for angles. You do **not** need to derive them for the practical.
- `sqrt` = square root; `arcsin` = inverse sine.
- `6371` = Earth's approximate radius in kilometres.
- `a` = an intermediate calculation in the formula.

### Formula

\[
a=\sin^2(\Delta\phi/2)+\cos(\phi_1)\cos(\phi_2)\sin^2(\Delta\lambda/2)
\]
\[
	ext{distance}=2	imes6371	imesrcsin(\sqrt a)
\]

**Memory trick:** Convert → Difference → Formula → Distance.

**Viva answer:** Haversine calculates great-circle distance using latitude and longitude. It is not the actual road route.

In [ ]:
# STEP A: Convert latitude angles from degrees to radians
lat1 = np.radians(df["pickup_latitude"])   # Pickup latitude
lat2 = np.radians(df["dropoff_latitude"])  # Drop-off latitude

# STEP B: Find differences between pickup and drop-off coordinates
dlat = lat2 - lat1
dlon = np.radians(df["dropoff_longitude"] - df["pickup_longitude"])

# STEP C: Apply the Haversine mathematical formula
# sin = sine, cos = cosine, **2 = square
a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2

# STEP D: Convert the result into kilometres (6371 = Earth radius in km)
df["distance"] = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

# Check the new distance column
print(df[["distance", "fare_amount"]].head())

## 6. Extract pickup hour and year

ML models need useful numeric inputs. The pickup time is a date-time string, so we extract:

- `.dt.hour` → hour of the day (0–23).
- `.dt.year` → year of the trip.

`pd.to_datetime()` converts text into date-time values. `errors="coerce"` turns unreadable dates into missing values so we can remove them.

In [ ]:
# Convert pickup date/time text into a datetime value
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")
df = df.dropna(subset=["pickup_datetime"])

# Extract two useful numbers from the date
df["hour"] = df["pickup_datetime"].dt.hour
df["year"] = df["pickup_datetime"].dt.year

# Original date column is no longer needed
df = df.drop(columns=["pickup_datetime"])
print(df[["hour", "year"]].head())

## 7. Remove invalid fares and passenger counts

We remove rows where:
- Fare is zero or negative.
- Distance is zero.
- Passenger count is not between 1 and 6.

These are **simple practical assumptions**, not rules for every taxi dataset.

In [ ]:
# Keep positive fare and distance
df = df[(df["fare_amount"] > 0) & (df["distance"] > 0)]

# Keep 1 to 6 passengers
df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 6)]
print("Rows after cleaning:", len(df))

## 8. Remove outliers using IQR

**Outlier** = an unusually small or large value (example: a very short trip with an extremely high fare).

**IQR = Interquartile Range**
- `Q1` = 25th percentile (25% of values are below it).
- `Q3` = 75th percentile.
- `IQR = Q3 − Q1`.
- Keep values between `Q1 − 1.5×IQR` and `Q3 + 1.5×IQR`.

`for col in [...]` repeats the same process for fare and distance. This is a common classroom method; it may remove some legitimate long trips.

In [ ]:
# Apply IQR filtering to fare and distance
for col in ["fare_amount", "distance"]:
    q1 = df[col].quantile(0.25)  # 25th percentile
    q3 = df[col].quantile(0.75)  # 75th percentile
    iqr = q3 - q1                # Middle 50% spread

    # Keep values inside the IQR limits
    df = df[(df[col] >= q1 - 1.5 * iqr) &
            (df[col] <= q3 + 1.5 * iqr)]

print("Rows after outlier removal:", len(df))

## 9. Correlation heatmap

**Correlation** shows how two numeric columns move together.

- Near **+1**: strong positive linear relationship.
- Near **−1**: strong negative linear relationship.
- Near **0**: weak/no *linear* relationship.

`corr()` calculates correlation. `sns.heatmap()` displays it as a coloured chart. `annot=True` prints the numbers on the chart.

In [ ]:
# Show correlation between the inputs and fare
columns = ["fare_amount", "distance", "passenger_count", "hour", "year"]
sns.heatmap(df[columns].corr(), annot=True, cmap="coolwarm")
plt.title("Uber Fare Correlation")
plt.show()

## 10. Select input (X) and output (y)

**X = Questions (features)**: distance, passengers, hour, year.

**y = Answer (target)**: actual fare.

Why `[[...]]` for X? → It creates a table with multiple input columns. `df["fare_amount"]` selects one output column.

In [ ]:
# Input features used for predicting the fare
X = df[["distance", "passenger_count", "hour", "year"]]

# Correct answers that the model will learn to predict
y = df["fare_amount"]

## 11. Split into training and testing data

- **80% training**: model learns from these trips.
- **20% testing**: model predicts fares it has not seen during training.
- `test_size=0.2` means 20% test.
- `random_state=42` makes the split repeatable; 42 is just a chosen number.

**Memory trick:** Train = Study, Test = Exam.

In [ ]:
# Divide data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 12. Train and evaluate both ML models

### Linear Regression
Finds a best-fitting **straight-line relationship** between inputs and fare.

### Random Forest Regression
Uses **many decision trees** and averages their predictions. `n_estimators=10` means 10 trees; `max_depth=10` limits tree complexity to make training faster.

### Three essential functions
- `fit(X_train, y_train)` = **learn** from training examples.
- `predict(X_test)` = **guess fares** for unseen trips.
- `r2_score()` and `mean_squared_error()` = **measure performance**.

**R²:** Higher is generally better (1 is perfect; negative values are possible).

**RMSE:** Root Mean Squared Error, in the same currency units as fare. Lower is better.

`def evaluate(...)` creates a reusable function so we don't write the same training code twice.

In [ ]:
# This function works for either ML algorithm
def evaluate(name, model):
    model.fit(X_train, y_train)       # Teach the model using training data
    pred = model.predict(X_test)      # Predict fares for testing data

    # Compare predictions with actual fares
    print(name)
    print("R2 Score:", r2_score(y_test, pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
    print()

# Model 1: Linear Regression (straight-line relationship)
evaluate("Linear Regression", LinearRegression())

# Model 2: Random Forest (combines multiple decision trees)
evaluate("Random Forest", RandomForestRegressor(
    n_estimators=10,     # Number of trees
    max_depth=10,        # Maximum levels in each tree
    random_state=42,     # Repeatable results
    n_jobs=-1            # Use available CPU cores
))

## 13. How to interpret your output

You'll see two blocks, one for each algorithm:

```text
Linear Regression
R2 Score: ...
RMSE: ...

Random Forest
R2 Score: ...
RMSE: ...
```

The numbers depend on the dataset and cleaning. **Do not memorize made-up scores.**

**How to compare:** Prefer a model with **higher test R²** and **lower test RMSE**. If the metrics disagree, explain the tradeoff rather than declaring an automatic winner.

> **Exam note:** This simple teaching notebook filters outliers before splitting the data, as in your original practical workflow. For production ML, learn filtering rules from training data only to reduce leakage.

## 14. Viva — quick answers to memorize

| Question | Easy answer |
|---|---|
| What is the aim? | Predict Uber fare from trip information. |
| What is regression? | Predicting a numerical value. |
| What is Haversine? | Distance between two points on Earth's surface from latitude and longitude. |
| What is latitude? | North–south geographical coordinate. |
| What is longitude? | East–west geographical coordinate. |
| Why 6371? | Earth's approximate radius in km. |
| What is an outlier? | An unusually high or low value. |
| What is IQR? | Q3 minus Q1, the spread of the middle 50% of values. |
| What is correlation? | Strength and direction of a linear relationship. |
| What is X? | Input features. |
| What is y? | Target fare to predict. |
| What is `fit()`? | Train the model. |
| What is `predict()`? | Predict outputs for new inputs. |
| What is Linear Regression? | Predicts using a fitted linear relationship. |
| What is Random Forest? | Combines predictions of multiple decision trees. |
| What is R²? | How well the model explains variation relative to a mean baseline. |
| What is RMSE? | Root mean squared prediction error; lower is better. |

### Final memory sequence
**Import → Load → Clean → Coordinates → Haversine → Time → Invalid rows → IQR → Heatmap → X/y → Split → Fit → Predict → Scores.**

**Tip:** Run every code cell once before your practical. If `uber.csv` cannot be found, open the folder containing both the notebook and CSV in VS Code.